# Tagging and Extraction Using OpenAI functions

In [1]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

In [2]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.utils.function_calling import convert_to_openai_function

In [3]:
class Tagging(BaseModel):
    """Tag the piece of text with particular info."""
    sentiment: str = Field(description="sentiment of text, should be `pos`, `neg`, or `neutral`")
    language: str = Field(description="language of text (should be in full text. Ex: English, Vietnamese, etc.)")

In [4]:
convert_to_openai_function(Tagging)

{'name': 'Tagging',
 'description': 'Tag the piece of text with particular info.',
 'parameters': {'properties': {'sentiment': {'description': 'sentiment of text, should be `pos`, `neg`, or `neutral`',
    'type': 'string'},
   'language': {'description': 'language of text (should be in full text. Ex: English, Vietnamese, etc.)',
    'type': 'string'}},
  'required': ['sentiment', 'language'],
  'type': 'object'}}

In [8]:
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain_fireworks import ChatFireworks
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI


In [9]:
model = ChatGroq(model_name="llama3-groq-70b-8192-tool-use-preview", temperature=0)
# model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
# model = ChatGoogleGenerativeAI(model="gemini-1.0-pro", temperature=0)

In [10]:
tagging_functions = [convert_to_openai_function(Tagging)]

In [11]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Think carefully, and then tag the text as instructed"),
    ("user", "{input}")
])

In [13]:
model_with_functions = model.bind(
    functions=tagging_functions,
    function_call={"name": "Tagging"}
)

In [15]:
tagging_chain = prompt | model_with_functions

In [16]:
tagging_chain.invoke({"input": "I love langchain"})

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"language": "English", "sentiment": "pos"}', 'name': 'Tagging'}}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 286, 'total_tokens': 304, 'completion_time': 0.054863421, 'prompt_time': 0.020851247, 'queue_time': 0.002889704, 'total_time': 0.075714668}, 'model_name': 'llama3-groq-70b-8192-tool-use-preview', 'system_fingerprint': 'fp_ee4b521143', 'finish_reason': 'function_call', 'logprobs': None}, id='run-6f9e7865-3242-4666-90a9-adbf631a6e03-0', usage_metadata={'input_tokens': 286, 'output_tokens': 18, 'total_tokens': 304})

In [19]:
tagging_chain.invoke({"input": "non mi piace questo cibo"})

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"language": "Italian", "sentiment": "neg"}', 'name': 'Tagging'}}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 289, 'total_tokens': 307, 'completion_time': 0.055257609, 'prompt_time': 0.032070387, 'queue_time': 0.0017002509999999998, 'total_time': 0.087327996}, 'model_name': 'llama3-groq-70b-8192-tool-use-preview', 'system_fingerprint': 'fp_ee4b521143', 'finish_reason': 'function_call', 'logprobs': None}, id='run-dccf6eaf-2d3f-469d-b569-cd8ce0dfc0b8-0', usage_metadata={'input_tokens': 289, 'output_tokens': 18, 'total_tokens': 307})

In [20]:
from langchain.output_parsers.openai_functions import JsonOutputFunctionsParser

In [21]:
tagging_chain = prompt | model_with_functions | JsonOutputFunctionsParser()

In [22]:
tagging_chain.invoke({"input": "non mi piace questo cibo"})

{'language': 'Italian', 'sentiment': 'neg'}

# Extraction

Extraction is similar to tagging, but used for extracting multiple pieces of information.

In [23]:
from typing import Optional
class Person(BaseModel):
    """Information about a person."""
    name: str = Field(description="person's name")
    age: Optional[int] = Field(description="person's age")

In [24]:
class Information(BaseModel):
    """Information to extract."""
    people: List[Person] = Field(description="List of info about people")

In [25]:
convert_to_openai_function(Information)

{'name': 'Information',
 'description': 'Information to extract.',
 'parameters': {'properties': {'people': {'description': 'List of info about people',
    'items': {'description': 'Information about a person.',
     'properties': {'name': {'description': "person's name", 'type': 'string'},
      'age': {'anyOf': [{'type': 'integer'}, {'type': 'null'}],
       'description': "person's age"}},
     'required': ['name', 'age'],
     'type': 'object'},
    'type': 'array'}},
  'required': ['people'],
  'type': 'object'}}

In [26]:
extraction_functions = [convert_to_openai_function(Information)]
extraction_model = model.bind(functions=extraction_functions, function_call={"name": "Information"})

In [27]:
extraction_model.invoke("Joe is 30, his mom is Martha")

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"people": [{"name": "Joe", "age": 30}, {"name": "Martha", "age": null}]}', 'name': 'Information'}}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 332, 'total_tokens': 365, 'completion_time': 0.103600943, 'prompt_time': 0.02414943, 'queue_time': 0.001014079000000001, 'total_time': 0.127750373}, 'model_name': 'llama3-groq-70b-8192-tool-use-preview', 'system_fingerprint': 'fp_ee4b521143', 'finish_reason': 'function_call', 'logprobs': None}, id='run-c7211278-6b51-423b-bb6f-8cd6104bf9e0-0', usage_metadata={'input_tokens': 332, 'output_tokens': 33, 'total_tokens': 365})

In [28]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract the relevant information, if not explicitly provided do not guess. Extract partial info"),
    ("human", "{input}")
])

In [29]:
extraction_chain = prompt | extraction_model

In [30]:
extraction_chain.invoke({"input": "Joe is 30, his mom is Martha"})

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"people": [{"name": "Joe", "age": 30}, {"name": "Martha", "age": null}]}', 'name': 'Information'}}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 348, 'total_tokens': 381, 'completion_time': 0.103322485, 'prompt_time': 0.027409459, 'queue_time': 0.0010841289999999996, 'total_time': 0.130731944}, 'model_name': 'llama3-groq-70b-8192-tool-use-preview', 'system_fingerprint': 'fp_ee4b521143', 'finish_reason': 'function_call', 'logprobs': None}, id='run-bbdd02e4-e379-4c8d-94bf-6bfcfc8f5475-0', usage_metadata={'input_tokens': 348, 'output_tokens': 33, 'total_tokens': 381})

In [31]:
extraction_chain = prompt | extraction_model | JsonOutputFunctionsParser()

In [32]:
extraction_chain.invoke({"input": "Joe is 30, his mom is Martha"})

{'people': [{'name': 'Joe', 'age': 30}, {'name': 'Martha', 'age': None}]}

In [ ]:
from langchain.output_parsers.openai_functions import JsonKeyOutputFunctionsParser

In [ ]:
extraction_chain = prompt | extraction_model | JsonKeyOutputFunctionsParser(key_name="people")

In [ ]:
extraction_chain.invoke({"input": "Joe is 30, his mom is Martha"})

# Doing it for real

We can apply tagging to a larger body of text.

For example, let's load this blog post and extract tag information from a sub-set of the text.

In [36]:
from langchain.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://www.promptingguide.ai/techniques/cot")
documents = loader.load()

In [34]:
doc = documents[0]

In [37]:
page_content = doc.page_content[:10000]

In [38]:
print(page_content[:1000])

Chain-of-Thought Prompting | Prompt Engineering Guide Prompt Engineering Guide🎓 Prompt Engineering Course🎓 Prompt Engineering CourseServicesServicesAboutAboutGitHubGitHub (opens in a new tab)DiscordDiscord (opens in a new tab)Prompt EngineeringIntroductionLLM SettingsBasics of PromptingPrompt ElementsGeneral Tips for Designing PromptsExamples of PromptsTechniquesZero-shot PromptingFew-shot PromptingChain-of-Thought PromptingMeta PromptingSelf-ConsistencyGenerate Knowledge PromptingPrompt ChainingTree of ThoughtsRetrieval Augmented GenerationAutomatic Reasoning and Tool-useAutomatic Prompt EngineerActive-PromptDirectional Stimulus PromptingProgram-Aided Language ModelsReActReflexionMultimodal CoTGraph PromptingGuidesOptimizing PromptsApplicationsFine-tuning GPT-4oFunction CallingContext Caching with LLMsGenerating DataGenerating Synthetic Dataset for RAGTackling Generated Datasets DiversityGenerating CodeGraduate Job Classification Case StudyPrompt FunctionPrompt HubClassificationSentim

In [40]:
class Overview(BaseModel):
    """Overview of a section of text."""
    summary: str = Field(description="Provide a concise summary of the content.")
    language: str = Field(description="Provide the language that the content is written in.")
    keywords: str = Field(description="Provide keywords related to the content.")

In [41]:
# from langchain_groq import ChatGroq
# model = ChatGroq(model_name="llama3-groq-70b-8192-tool-use-preview", temperature=0)
overview_tagging_function = [
    convert_to_openai_function(Overview)
]
tagging_model = model.bind(
    functions=overview_tagging_function,
    function_call={"name":"Overview"}
)
tagging_chain = prompt | tagging_model | JsonOutputFunctionsParser()

open ai

In [ ]:
tagging_chain.invoke({"input": page_content})

llama 70B

In [42]:
tagging_chain.invoke({"input": page_content})

{'keywords': 'Chain-of-Thought Prompting, Zero-shot COT Prompting, Automatic Chain-of-Thought, Auto-CoT, question clustering, demonstration sampling',
 'language': 'English',
 'summary': 'This section discusses various techniques in prompt engineering, including Chain-of-Thought (CoT) Prompting, Zero-shot COT Prompting, and Automatic Chain-of-Thought (Auto-CoT). CoT Prompting enables complex reasoning capabilities through intermediate reasoning steps. Zero-shot COT Prompting uses a simple prompt to improve reasoning. Auto-CoT is an automatic process that leverages LLMs to generate reasoning chains for demonstrations, eliminating manual efforts. It consists of question clustering and demonstration sampling stages.'}

In [43]:
class Paper(BaseModel):
    """Information about papers mentioned."""
    title: str
    author: Optional[str]


class Info(BaseModel):
    """Information to extract"""
    papers: List[Paper]

In [44]:
template = """A article will be passed to you. Extract from it all papers that are mentioned by this article. 

Do not extract the name of the article itself. If no papers are mentioned that's fine - you don't need to extract any! Just return an EMPTY LIST.

Do not make up or guess ANY extra information. Only extract what exactly is in the text.

if the input is irrelevant, return an empty list."""

prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("human", "{input}")
])

In [46]:
from langchain_community.output_parsers.ernie_functions import JsonKeyOutputFunctionsParser
extraction_functions = [convert_to_openai_function(Info)]
extraction_model = model.bind(functions=extraction_functions, function_call={"name":"Info"})
extraction_chain = prompt | extraction_model | JsonKeyOutputFunctionsParser(key_name="papers")

In [47]:
extraction_chain.invoke({"input": page_content})

[{'title': 'Chain-of-Thought Prompting', 'author': 'Wei et al. (2022)'},
 {'title': 'Zero-shot CoT Prompting', 'author': 'Kojima et al. (2022)'},
 {'title': 'Automatic Chain-of-Thought (Auto-CoT)',
  'author': 'Zhang et al. (2022)'}]

In [ ]:
extraction_chain.invoke({"input": "hi"})

Split into pieces of texts and pass to the LLM and combine all the results of the end

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_overlap=0)

In [ ]:
splits = text_splitter.split_text(doc.page_content)

In [ ]:
len(splits)

In [ ]:
# Function to flatten a 2D matrix into a 1D list
def flatten(matrix):
    flat_list = []
    for row in matrix:
        flat_list += row
    return flat_list

In [ ]:
from langchain.schema.runnable import RunnableLambda

In [ ]:
# This RunnableLambda prepares the input for extraction by splitting it into chunks
# and creating a list of dictionaries, each containing a chunk as the "input" value
prep = RunnableLambda(
    lambda x: [{"input": doc} for doc in text_splitter.split_text(x)]
)

In [ ]:
prep.invoke("hi")

In [ ]:
chain = prep | extraction_chain.map() | flatten

In [ ]:
chain.invoke(doc.page_content)

# Conversational Agent

## Tool and Routing

In [ ]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']

Tool get weather

In [ ]:
from langchain.agents import tool

import requests
from pydantic import BaseModel, Field
import datetime

# Define the input schema
class OpenMeteoInput(BaseModel):
    latitude: float = Field(..., description="Latitude of the location to fetch weather data for")
    longitude: float = Field(..., description="Longitude of the location to fetch weather data for")

@tool(args_schema=OpenMeteoInput)
def get_current_temperature(latitude: float, longitude: float) -> dict:
    """Fetch current temperature for given coordinates."""    
    BASE_URL = "https://api.open-meteo.com/v1/forecast"
    
    # Parameters for the request
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'hourly': 'temperature_2m',
        'forecast_days': 1,
    }

    # Make the request
    response = requests.get(BASE_URL, params=params)
    
    if response.status_code == 200:
        results = response.json()
    else:
        raise Exception(f"API Request failed with status code: {response.status_code}")

    current_utc_time = datetime.datetime.utcnow()
    time_list = [datetime.datetime.fromisoformat(time_str.replace('Z', '+00:00')) for time_str in results['hourly']['time']]
    temperature_list = results['hourly']['temperature_2m']
    
    closest_time_index = min(range(len(time_list)), key=lambda i: abs(time_list[i] - current_utc_time))
    current_temperature = temperature_list[closest_time_index]
    
    return f'The current temperature is {current_temperature}°C'

In [ ]:
print(get_current_temperature.name)
print(get_current_temperature.args)
print(get_current_temperature.description)


Tool search wikipedia

In [ ]:
import wikipedia
@tool
def search_wikipedia(query: str) -> str:
    """Run Wikipedia search and get page summaries."""
    page_titles = wikipedia.search(query)
    summaries = []
    for page_title in page_titles[:3]:
        try:
            wiki_page = wikipedia.page(title=page_title, auto_suggest=False)
            summaries.append(f"Page: {page_title}\nSummary: {wiki_page.summary}")
        except (
            wikipedia.exceptions.PageError,
            wikipedia.exceptions.DisambiguationError,
        ):
            pass
    if not summaries:
        return "No good Wikipedia Search Result was found"
    return "\n\n".join(summaries)

In [ ]:
print(search_wikipedia.name)
print(search_wikipedia.args)
print(search_wikipedia.description)

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_fireworks import ChatFireworks
from langchain.prompts import ChatPromptTemplate
from langchain.tools.render import format_tool_to_openai_function
from langchain.agents.output_parsers import OpenAIFunctionsAgentOutputParser
from langchain_core.utils.function_calling import convert_to_openai_function
from langchain_google_genai import ChatGoogleGenerativeAI


In [ ]:
functions = [
    convert_to_openai_function(f) for f in [
        search_wikipedia, get_current_temperature
    ]
]
# model = ChatFireworks(model="accounts/fireworks/models/llama-v3p1-70b-instruct",temperature=0).bind(functions=functions)
model = ChatGroq(model_name="llama3-groq-70b-8192-tool-use-preview", temperature=0).bind(functions=functions)
# model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0).bind(functions=functions)
# model = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0).bind(functions=functions)

In [ ]:
result = model.invoke("what is the weather in HCM city right now")
result

In [ ]:
from langchain.schema.agent import AgentFinish
def parse_output(output):
    if isinstance(output, AgentFinish):
        return output.return_values['output']
    else:
        tools = {
            "search_wikipedia": search_wikipedia, 
            "get_current_temperature": get_current_temperature,
        }
        return tools[output.additional_kwargs["tool_calls"][0]["function"]["name"]].invoke(eval(output.additional_kwargs["tool_calls"][0]["function"]["arguments"]))


In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful but sassy assistant. use provided tools to answer user questions, your response shoulde follow the format of the tool"),
    ("user", "{input}"),
])
chain = prompt | model | parse_output
results = chain.invoke({"input": "what is the weather in HCM city right now"})
print(results)

In [ ]:
get_current_temperature.invoke({"latitude": 10.762622, "longitude": 106.660172})

In [ ]:
results = chain.invoke({"input": "Tell me about Doremon."})
print(results)

In [ ]:
print(search_wikipedia.invoke({"query": "Doremon"}))

## Agents with LangChain

The agent will take the query, run the tools and to get the result. Then from the result, it will create a new query to run the tools again untill it get the final result or exceed the max_iter


In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.agents.format_scratchpad import format_to_openai_functions
from langchain.agents.output_parsers import OpenAIFunctionsAgentOutputParser
from langchain.prompts import MessagesPlaceholder
from langchain.schema.runnable import RunnablePassthrough
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.memory import ConversationBufferMemory

In [ ]:

model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
# model = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful assistant. use provided tools to answer user questions."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

memory = ConversationBufferMemory(return_messages=True,memory_key="chat_history")

agent_chain = RunnablePassthrough.assign(
    agent_scratchpad= lambda x: format_to_openai_functions(x["intermediate_steps"])
) | prompt | model | OpenAIFunctionsAgentOutputParser()

In [ ]:
from langchain.agents import AgentExecutor
tools = [get_current_temperature, search_wikipedia]
agent_executor = AgentExecutor(agent=agent_chain, tools=tools, verbose=True, memory=memory)

In [ ]:
agent_executor.invoke({"input": "Tell me about gamer Faker."})

In [ ]:
agent_executor.invoke({"input": "what is the weather in HCM city right now"})

In [ ]:
agent_executor.invoke({"input": "hi"})